# TP0 — Anatomía de un frame (informe)

**Nombre:** Matías · **Fecha de entrega:** _completar_

Este notebook **usa** tu librería (`mv.imagen`, `mv.cinematica`). Las funciones se implementan en `mv/`, no acá.
Completá el código marcado con `# TODO` y respondé las preguntas en las celdas **✍️ Respuesta**.

> Si una celda tira `NotImplementedError`, es porque todavía no implementaste esa función en `mv/`.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

from mv import imagen as im
from mv import cinematica as cin
from mv.datos import cargar_frame, mostrar, trayectoria_ejemplo, listar_frames

print("Frames reales disponibles:", len(listar_frames()))

## 1. Anatomía

Cargamos un frame (real si hay en `datos/frames/`, sintético si no).

In [ ]:
img = cargar_frame()          # RGB, uint8
print("shape:", img.shape, "| dtype:", img.dtype, "| MB:", img.nbytes / 1e6)
print("píxel de arriba a la izquierda:", img[0, 0])
mostrar(img, titulos=["frame"], ancho=10)

**Preguntas**
1. ¿Cuánto mide el frame de alto y de ancho? ¿Cuántos MB ocuparían 90 minutos a 25 fps sin comprimir?
2. Elegí un punto que se vea en la imagen (por ejemplo, la pelota o la cabeza de un jugador) y escribí sus coordenadas `(x, y)` y cómo leerías su color en el array.

✍️ **Respuesta:**

In [ ]:
# Los tres canales por separado y el gris
gris = im.a_gris(img)
mostrar(img[..., 0], img[..., 1], img[..., 2], gris,
        titulos=["R", "G", "B", "gris (0.299, 0.587, 0.114)"], ancho=5)

**Preguntas**
3. ¿En qué canal el pasto se ve más claro? ¿Y las camisetas de cada equipo?
4. El gris le da 0,587 de peso al verde y solo 0,114 al azul. ¿Por qué crees que es así? (Pista: no tiene que ver con las cámaras sino con el ojo humano.)

✍️ **Respuesta:**

## 2. Histogramas

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for k, (nombre, color) in enumerate([("R", "tab:red"), ("G", "tab:green"), ("B", "tab:blue")]):
    ax.plot(im.histograma(img[..., k]), color=color, label=nombre)
ax.set_xlabel("valor (0–255)"); ax.set_ylabel("cantidad de píxeles"); ax.legend()
ax.set_title("Histograma por canal")
plt.show()

**Preguntas**
5. ¿Qué picos ves y a qué parte de la imagen corresponde cada uno? (Pasto, tribuna, líneas...)
6. ¿Hay píxeles saturados (en 0 o 255)? ¿Qué significaría eso para un algoritmo que quiera "ver" detalles en esa zona?

✍️ **Respuesta:**

## 3. Overflow, bien y mal

In [ ]:
mal = img + np.uint8(100)                              # suma "ingenua" en uint8
bien = im.ajustar_brillo_contraste(img, alfa=1.0, beta=100)
mostrar(img, mal, bien, titulos=["original", "img + 100 (uint8)", "ajustar_brillo_contraste"], ancho=6)
print("píxeles que 'dieron la vuelta':", (mal < img).sum())

**Preguntas**
7. Explicá qué pasó en la imagen del medio con un píxel concreto (valor antes y después).
8. ¿Por qué `mal < img` cuenta exactamente los píxeles que hicieron overflow?

✍️ **Respuesta:**

## 4. El pasto

In [ ]:
m = im.mascara_pasto(img)
print(f"fracción de pasto: {m.mean():.3f}")
print("color medio del pasto (RGB):", im.color_medio(img, m).round(1))
mostrar(img, m, titulos=["frame", "máscara de pasto"], ancho=7)

In [ ]:
# TODO: barrer el margen y graficar la fracción de pasto en función del margen
margenes = [0, 5, 10, 20, 30, 40, 60]
fracciones = ...   # una fracción por margen (podés usar un for sobre los márgenes: son 7, no píxeles)

plt.plot(margenes, fracciones, "o-"); plt.xlabel("margen"); plt.ylabel("fracción de pasto"); plt.grid(alpha=.3)
plt.show()

**Preguntas**
9. ¿Qué cosas que **no** son pasto entran en la máscara con margen 0? ¿Qué pasto se pierde con margen 60? (Mirá las máscaras, no solo el gráfico.)
10. ¿Qué limitación tiene una regla "G > R + margen" con sombras fuertes? (En U2 lo resolvemos con HSV.)

✍️ **Respuesta:**

## 5. Jugadores

Elegí **6 jugadores** (idealmente 3 de cada equipo) y escribí sus cajas `(x1, y1, x2, y2)`.

Para leer coordenadas: `plt.imshow(img)` con los ejes visibles, o abrí el frame en cualquier visor que muestre la posición del mouse.

In [ ]:
plt.figure(figsize=(14, 8)); plt.imshow(img); plt.grid(alpha=.3); plt.title("leé las coordenadas acá"); plt.show()

In [ ]:
# TODO: tus 6 cajas (x1, y1, x2, y2)
cajas = [
    # (x1, y1, x2, y2),
]

recortes = [im.recortar(img, c) for c in cajas]

# Para el mosaico todas tienen que tener la misma shape: las llevamos a 96×48 con OpenCV
# (redimensionar a mano lo hacemos en U1, con interpolación)
import cv2
iguales = [cv2.resize(r, (48, 96), interpolation=cv2.INTER_NEAREST) for r in recortes]
mostrar(im.mosaico(iguales, columnas=6), titulos=["los 6 jugadores"], ancho=10)

In [ ]:
# TODO: el color medio del TORSO de cada jugador (la mitad superior de la caja, sin la cabeza)
# y del pasto alrededor. Imprimilos en una tabla.

**Preguntas**
11. ¿Los colores medios de los torsos se separan claramente en dos grupos? ¿Qué jugador te costó más y por qué?
12. Si en vez del torso usaras la caja entera, ¿qué se mezclaría en el promedio?

✍️ **Respuesta:**

## 6. Vista contra copia: romperla a propósito

In [ ]:
img2 = img.copy()
zona = img2[100:200, 100:300]      # slicing: ¿vista o copia?
zona[:] = 0                         # pintamos de negro "la zona"
mostrar(img2, titulos=["¿qué le pasó a img2?"], ancho=8)
print("¿comparten memoria?", np.shares_memory(img2, zona))

**Preguntas**
13. Explicá qué pasó y por qué. ¿Cómo lo evitarías? ¿En qué caso querrías justamente este comportamiento?

✍️ **Respuesta:**

## 7. La trayectoria de un jugador

`trayectoria_ejemplo()` simula 60 s de un jugador (trote, piques, quieto), con ruido de medición y dos saltos de tracking.

In [ ]:
P, fps = trayectoria_ejemplo()
plt.figure(figsize=(7, 5)); plt.plot(P[:, 0], P[:, 1], ".-", ms=3); plt.axis("equal")
plt.xlabel("x (m)"); plt.ylabel("y (m)"); plt.title("trayectoria"); plt.show()

print(f"distancia cruda:     {cin.distancia_total(P):7.1f} m")
print(f"distancia filtrada:  {cin.distancia_total_filtrada(P, fps):7.1f} m")
print(f"velocidad máxima:    {cin.velocidades_kmh(P, fps).max():7.1f} km/h")
print(f"velocidad tope (p95):{cin.velocidad_tope(P, fps):7.1f} km/h")

In [ ]:
v = cin.velocidades_kmh(P, fps)
plt.figure(figsize=(10, 3)); plt.plot(np.arange(len(v)) / fps, v); plt.ylim(0, 60)
plt.xlabel("tiempo (s)"); plt.ylabel("km/h"); plt.title("velocidad (recortada en 60 km/h)"); plt.show()

**Preguntas**
14. ¿Cuánto infla la distancia cada uno de los dos saltos? ¿Coincide con lo que esperarías mirando el gráfico?

✍️ **Respuesta:**

In [ ]:
# Experimento: el ruido infla la distancia, y cuánto depende de cada cuántos frames medís.
# La trayectoria SIN saltos y SIN ruido es la "verdad"; comparamos contra la medida con ruido.
P_ruido, fps = trayectoria_ejemplo(con_saltos=False)

# TODO: para k = 1, 2, 3, 5, 10 tomá P_ruido[::k] (un punto de cada k) y calculá la distancia total.
# Graficá distancia vs k. ¿Qué pasa y por qué?

**Preguntas**
15. ¿Por qué con k = 1 la distancia da más que con k = 10? ¿Muestrear menos es entonces "mejor"? ¿Qué se pierde si k es muy grande? (Pensá en un jugador que zigzaguea.)
16. Con lo que viste: si tuvieras que reportarle a un DT "distancia recorrida por jugador", ¿qué dos cosas harías sí o sí antes de sumar?

✍️ **Respuesta:**

## Conclusiones

En 5–10 líneas: lo más importante que aprendiste en este TP, lo que todavía no te cierra y lo que harías distinto.

✍️ **Respuesta:**